# 05 — PCA + linear / kernel

Fit on train; eval attribution on **selection**. Pipeline includes ExpandingScaler.


In [ ]:
from spy_alpha_research.data import (
    load_config, project_root, load_frame, save_frame, split_three_way, make_target,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]
dates = exp["dates"]

from spy_alpha_research.pipelines import fit_pca_ridge, fit_pca_elasticnet, fit_pca_kernel_ridge
from spy_alpha_research.reporting import plot_importance

feats = load_frame(PROC / "features_raw.parquet")
y = load_frame(PROC / "spy_target.parquet").iloc[:, 0]
train_idx, sel_idx, final_idx = split_three_way(
    feats.index,
    train_end=dates["train_end"],
    selection_start=dates["selection_start"],
    selection_end=dates["selection_end"],
    final_start=dates["final_start"],
    end=dates["end"],
)
cfg = models["pca_kernel"]
mp = exp["scaling"]["min_periods"]
ridge = fit_pca_ridge(
    feats.loc[train_idx], y.reindex(train_idx),
    feats.loc[sel_idx], y.reindex(sel_idx),
    n_components=cfg["n_components"], alpha=cfg["ridge_alpha"], scale=True, min_periods=mp,
)
kr = fit_pca_kernel_ridge(
    feats.loc[train_idx], y.reindex(train_idx),
    feats.loc[sel_idx], y.reindex(sel_idx),
    n_components=cfg["n_components"], alpha=cfg["kernel_ridge_alpha"],
    gamma=cfg["kernel_ridge_gamma"], scale=True, min_periods=mp,
)
plot_importance(ridge.feature_attribution, FIG / "pca_loadings.png", title="PCA-Ridge (selection)")
plot_importance(kr.feature_attribution, FIG / "kernel_perm_importance.png", title="Kernel perm (selection)")
ridge.feature_attribution.head(10), kr.feature_attribution.head(10)
